###                                             **📘 JINJA IN SQL & DBT**

#### **1️⃣ What is Jinja in dbt?**

**Jinja** is a **templating engine** used by dbt to **generate SQL dynamically.**

📌 Execution order:

In [ ]:
Jinja → SQL → Database

dbt:

- Resolves Jinja

- Produces pure SQL

- Executes SQL on the database

---------------

**2️⃣ Core Jinja Syntax (Must Remember)**

**🔹 Expressions → `{{ }}`**

Used to **print / replace values**

In [ ]:
select '{{ 1 + 1 }}' as result

➡ Compiled SQL:

In [ ]:
select '2' as result

------

**🔹 Statements → `{% %}`**

Used for **logic**, not output

In [ ]:
{% if true %}
select 'yes'
{% endif %}

__________

**🔹 Comments → `{# #}`**

In [ ]:
{# ignored by dbt #}

-----------

#### **3️⃣ Why Jinja is Needed in SQL**

Without Jinja (❌):

- Hard-coded schemas

- No environment awareness

- Repeated logic

With Jinja (✅):

- Dynamic SQL

- Environment-safe

- Reusable logic

- Dependency tracking

--------------------

#### **4️⃣ dbt ref() — Model Reference**

**Purpose**

- References another dbt model

- Builds DAG

- Resolves schema automatically

**Example**

In [ ]:
select *
from {{ ref('stg_customers') }}

➡ Compiled SQL:

In [ ]:
select *
from dev.analytics.stg_customers

📌 Always use `ref()` for dbt models.

-------------

**5️⃣ dbt `source()` — Raw Tables**

Used for **raw/source data only.**

In [ ]:
select *
from {{ source('airbnb', 'listings') }}

➡ Compiled SQL:

In [ ]:
select *
from raw.raw_listings

Benefits:

- Freshness checks

- Source tests

- Clear data lineage


----------

#### **6️⃣ Variables — var()**

**Define in `dbt_project.yml`**

In [ ]:
vars:
  country: IN

**Use in SQL**

In [ ]:
where country = '{{ var("country") }}'

➡ Compiled SQL:

In [ ]:
where country = 'IN'

-----------

#### **7️⃣ Control Flow (IF / ELSE)**

Used for **conditional SQL generation.**

In [ ]:
select *
from {{ ref('orders') }}

{% if target.name == 'prod' %}
where created_at >= current_date - 30
{% else %}
where created_at >= current_date - 7
{% endif %}

📌 Same model → different SQL per environment.

---------------

#### **8️⃣ Loops (`for`)**

Used to **avoid repetitive SQL.**

In [ ]:
select
{% for col in ['id', 'name', 'email'] %}
  {{ col }}{% if not loop.last %},{% endif %}
{% endfor %}
from users

➡ Compiled SQL:

In [ ]:
select
  id,
  name,
  email
from users

-------------

#### **9️⃣ `config()` — Model Configuration (IMPORTANT)**

Used to configure **model behavior.**

**Syntax**

In [ ]:
{{ config(
    materialized='table',
    schema='analytics',
    tags=['core']
) }}

**Common Config Options**

| Config             | Purpose                                |
| ------------------ | -------------------------------------- |
| `materialized`     | table / view / incremental / ephemeral |
| `schema`           | Override schema                        |
| `alias`            | Rename model                           |
| `tags`             | Group models                           |
| `unique_key`       | Incremental models                     |
| `on_schema_change` | Schema handling                        |


-------------

**Example: Incremental Model**

In [ ]:
{{ config(
    materialized='incremental',
    unique_key='order_id'
) }}

select *
from {{ source('raw', 'orders') }}

{% if is_incremental() %}
where updated_at > (select max(updated_at) from {{ this }})
{% endif %}

------------

#### **🔟 `this` — Current Model Reference**

In [ ]:
{{ this }}

Resolves to:

In [ ]:
database.schema.model_name

Used mainly in:

- Incremental models

- Auditing logic

--------------

#### **1️⃣1️⃣ is_incremental()**

Returns **true** only when:

- Model exists

- Incremental run

In [ ]:
{% if is_incremental() %}
  where updated_at > (select max(updated_at) from {{ this }})
{% endif %}

-----------

**1️⃣2️⃣ Macros — Reusable SQL Logic**

**Define Macro**

In [ ]:
{% macro clean_text(col) %}
lower(trim({{ col }}))
{% endmacro %}

**Use Macro**

In [ ]:
select
  {{ clean_text('name') }} as name
from users

➡ Compiled SQL:

In [ ]:
select
  lower(trim(name)) as name
from users

----------

#### **1️⃣3️⃣ Adapter & Adapter Dispatch (IMPORTANT)**

**What is an Adapter?**

An **adapter** allows dbt to work with different databases.

Examples:

- Snowflake

- Postgres

- BigQuery

- Redshift

Each database has **different SQL syntax.**

---------------

**Adapter-aware Logic**

In [ ]:
{% if target.type == 'snowflake' %}
  current_timestamp()
{% elif target.type == 'postgres' %}
  now()
{% endif %}

-------------

**Adapter Dispatch (Advanced & Best Practice)**

Used when writing **database-specific macros.**

In [ ]:
{% macro current_ts() %}
  {{ adapter.dispatch('current_ts')() }}
{% endmacro %}

Snowflake version:

In [ ]:
{% macro snowflake__current_ts() %}
  current_timestamp()
{% endmacro %}

Postgres version:

In [ ]:
{% macro postgres__current_ts() %}
  now()
{% endmacro %}

dbt automatically picks the correct one.

------------

#### **1️⃣4️⃣ `target` Object**

In [ ]:
{{ target.name }}      -- dev / prod
{{ target.schema }}
{{ target.database }}
{{ target.type }}      -- snowflake / postgres

Used for:

- Environment logic

- Adapter logic

---------------

#### **1️⃣5️⃣ Compilation (VERY IMPORTANT)**

**Command**

In [ ]:
dbt compile

**What it does:**

- Resolves all Jinja

- Outputs final SQL in:

In [ ]:
target/compiled/

📌 Best way to **learn & debug Jinja.**

-------------

**1️⃣6️⃣ Best Practices ⭐**

✅ Always use `ref()` and `source()`

✅ Use `config()` at top of model

✅ Keep Jinja minimal & readable

✅ Use macros for repeated logic

✅ Test compiled SQL

❌ Avoid business logic in Jinja


-----------------

#### **🔚 FINAL QUICK SUMMARY**

| Concept            | Purpose           |
| ------------------ | ----------------- |
| Jinja              | SQL generation    |
| `{{ }}`            | Output value      |
| `{% %}`            | Logic             |
| `ref()`            | dbt model         |
| `source()`         | Raw table         |
| `config()`         | Model config      |
| `this`             | Current model     |
| `is_incremental()` | Incremental logic |
| Adapter            | DB-specific SQL   |
| Macros             | Reusable SQL      |
